# Ham ya da Spam?

🎯 Bu görevin amacı, e-postaları **spam (1)** veya **normal e-posta (0)** olarak sınıflandırmaktır.

🧹 İlk olarak, bu metin verilerine **temizleme (cleaning)** teknikleri uygulanacaktır.

👩🏻‍🔬 Ardından, temizlenmiş metinler **sayısal bir gösterime** dönüştürülecektir.

✉️ Son olarak, her bir e-postayı spam mı yoksa normal mi olduğunu sınıflandırmak için  
***Multinomial Naive Bayes*** modeli uygulanacaktır.


## (0) NTLK kütüphanesi (Doğal Dil Araç Seti)

In [1]:
!pip install nltk


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [10]:
# nltk'yi ilk kez içe aktarırken, birkaç yerleşik kütüphaneyi de indirmemiz gerekir.

import nltk

nltk.download('stopwords')
nltk.download('punkt')      # nltk<3.9.0 için
nltk.download('punkt_tab')  # nltk>=3.9.0 için
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /home/mert/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/mert/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/mert/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /home/mert/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/mert/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [3]:
import pandas as pd

df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/ham_spam_emails.csv")
df.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


## (1) (Metin) veri setinin temizlenmesi

Veri kümesi, ham [0] veya spam [1] olarak sınıflandırılan e-postalardan oluşur. Tahmin modelini eğitmeden önce veri kümesini temizlemeniz gerekir.

### (1.1) Noktalama İşaretlerini Kaldır

❓ Noktalama işaretlerini kaldırmak için bir işlev oluşturun. Bunu `text` sütununa uygulayın ve çıktıyı `clean_text` adlı veri çerçevesinin yeni bir sütununa ekleyin. ❓

In [6]:
import string

# Noktalama işaretlerini kaldıran fonksiyon
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

# text sütununa uygulama
df["clean_text"] = df["text"].apply(remove_punctuation)

# İlk birkaç satırı görüntüleme
df.head()

,text,spam,clean_text
0,Subject: naturally irresistible your corporate...,1,Subject naturally irresistible your corporate ...
1,Subject: the stock trading gunslinger fanny i...,1,Subject the stock trading gunslinger fanny is...
2,Subject: unbelievable new homes made easy im ...,1,Subject unbelievable new homes made easy im w...
3,Subject: 4 color printing special request add...,1,Subject 4 color printing special request addi...
4,"Subject: do not have money , get software cds ...",1,Subject do not have money get software cds fr...


### (1.2) Küçük Harf

❓ Metni küçük harfe çeviren bir işlev oluşturun. Bunu `clean_text`'e uygulayın ❓

In [7]:
# Metni küçük harfe çeviren fonksiyon
def to_lowercase(text):
    return text.lower()

# clean_text sütununa uygulama
df["clean_text"] = df["clean_text"].apply(to_lowercase)

# İlk birkaç satırı görüntüleme
df.head()

,text,spam,clean_text
0,Subject: naturally irresistible your corporate...,1,subject naturally irresistible your corporate ...
1,Subject: the stock trading gunslinger fanny i...,1,subject the stock trading gunslinger fanny is...
2,Subject: unbelievable new homes made easy im ...,1,subject unbelievable new homes made easy im w...
3,Subject: 4 color printing special request add...,1,subject 4 color printing special request addi...
4,"Subject: do not have money , get software cds ...",1,subject do not have money get software cds fr...


### (1.3) Sayıları Kaldır

❓ Metinden sayıları kaldırmak için bir işlev oluşturun. Bunu `clean_text`'e uygulayın ❓

In [8]:
import re

# Sayıları kaldıran fonksiyon
def remove_numbers(text):
    return re.sub(r'\d+', '', text)

# clean_text sütununa uygulama
df["clean_text"] = df["clean_text"].apply(remove_numbers)

# İlk birkaç satırı görüntüleme
df.head()

,text,spam,clean_text
0,Subject: naturally irresistible your corporate...,1,subject naturally irresistible your corporate ...
1,Subject: the stock trading gunslinger fanny i...,1,subject the stock trading gunslinger fanny is...
2,Subject: unbelievable new homes made easy im ...,1,subject unbelievable new homes made easy im w...
3,Subject: 4 color printing special request add...,1,subject color printing special request addit...
4,"Subject: do not have money , get software cds ...",1,subject do not have money get software cds fr...


### (1.4) StopWords'ü kaldırın

❓ Metinden durdurma kelimelerini kaldırmak için bir işlev oluşturun. Bunu `clean_text`'e uygulayın. ❓

In [11]:
from nltk.corpus import stopwords
import nltk

stop_words = set(stopwords.words('english'))

# Durdurma kelimelerini kaldıran fonksiyon
def remove_stopwords(text):
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return " ".join(filtered_words)

# clean_text sütununa uygulama
df["clean_text"] = df["clean_text"].apply(remove_stopwords)

# İlk birkaç satırı görüntüleme
df.head()

,text,spam,clean_text
0,Subject: naturally irresistible your corporate...,1,subject naturally irresistible corporate ident...
1,Subject: the stock trading gunslinger fanny i...,1,subject stock trading gunslinger fanny merrill...
2,Subject: unbelievable new homes made easy im ...,1,subject unbelievable new homes made easy im wa...
3,Subject: 4 color printing special request add...,1,subject color printing special request additio...
4,"Subject: do not have money , get software cds ...",1,subject money get software cds software compat...


### (1.5) Lemmatize

❓ Metni lemmatize etmek için bir fonksiyon oluşturun. Çıktının bir kelime listesi değil, tek bir dize olduğundan emin olun. Bunu `clean_text`'e uygulayın. ❓

In [19]:
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

lemmatizer = WordNetLemmatizer()

# Lemmatization fonksiyonu
def lemmatize_text(text):
    words = word_tokenize(text)
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    
    # Listeyi tekrar tek bir string haline getir
    return " ".join(lemmatized_words)

# clean_text sütununa uygulama
df["clean_text"] = df["clean_text"].apply(lemmatize_text)

# İlk birkaç satırı görüntüleme
df.head()

,text,spam,clean_text
0,Subject: naturally irresistible your corporate...,1,subject naturally irresistible corporate ident...
1,Subject: the stock trading gunslinger fanny i...,1,subject stock trading gunslinger fanny merrill...
2,Subject: unbelievable new homes made easy im ...,1,subject unbelievable new home made easy im wan...
3,Subject: 4 color printing special request add...,1,subject color printing special request additio...
4,"Subject: do not have money , get software cds ...",1,subject money get software cd software compati...


## (2) Bag-of-Words Modellemesi

### (2.1) Metin verilerini sayılara dönüştürme

❓ `clean_text`'i varsayılan CountVectorizer ile Bag-of-Words temsiline vektörleştirin. `X_bow` olarak kaydedin. ❓

In [20]:
from sklearn.feature_extraction.text import CountVectorizer

# CountVectorizer oluştur
vectorizer = CountVectorizer()

# clean_text sütununu Bag-of-Words formatına dönüştür
X_bow = vectorizer.fit_transform(df["clean_text"])

# Boyut kontrolü
print(X_bow.shape)

(5728, 30974)


### (2.2) Çok terimli Naive Bayes Modellemesi

❓ MultinomialNB modelini bag-of-words verileriyle çapraz doğrulayın. Modelin doğruluğunu puanlayın. ❓

In [22]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_score

model = MultinomialNB()

scores = cross_val_score(model, X_bow, df.spam, cv=5, scoring="accuracy")

print("Doğruluk skorları:", scores)
print("Ortalama doğruluk:", scores.mean())

Doğruluk skorları: [0.98691099 0.9895288  0.991274   0.98777293 0.99213974]
Ortalama doğruluk: 0.9895252901681946


🏁 Tebrikler!

💾 Not defterinizi git add/commit/push yapmayı unutmayın...

🚀 ... ve bir sonraki challenge'a geçin!